# Master pipeline v4 — headway distributions & compound AFT models (`data3`)

**What this notebook does**

| Step | Cell | Output |
|---|---|---|
| 1. Inferential statistics (Overall / BTW / PR) | 4 | `Tables/01_inferential_stats.xlsx` |
| 2. Multicollinearity diagnostics — **report only** | 5 | `Tables/02_vif_multicollinearity.xlsx`, `fig02a/b` |
| 3. Distribution fitting per pair (all candidates) | 6–7 | `Tables/03_distribution_fits.xlsx` |
| 4. Pair retention (rule-based) | 8 | printed + `00_provenance.xlsx` |
| 5. Univariate covariate screen under each pair's own best distribution | 9–10 | `Tables/04_covariate_screen.xlsx`, `fig04` |
| 6. Final covariate set (VIF reported, never applied) | 11 | `Tables/05_final_covariate_sets.xlsx` |
| 7. **Multivariable compound AFT refit** | 12 | `Tables/06_compound_models.xlsx`, `fig06` |
| 8. Fit diagnostics figures | 13 | `fig03a/b/c` |
| 9. Provenance / manifest | 14 | `Tables/00_provenance.xlsx` |

**Covariate policy (per instruction).** Every column enters the covariate set except
the outcome (`Time_Headway`), the pair-defining classes (`V_Target`, `V_Leading_Class`,
`Pair`), and `Leading_Speed_km/hr`. Leading speed is excluded because
`Speed_Difference = Target_Speed − Leading_Speed` holds *exactly* in these data, so the
three together are perfectly collinear — including all of them would make the design
matrix singular.

**VIF policy (per instruction).** Cells 5 and 11 compute and report VIF, tolerance,
$R^2_j$, the Spearman correlation matrix, the design-matrix condition number, and
per-pair VIFs. **No covariate is ever removed on the basis of VIF.**

---

## Issues found in v3 and how they are fixed

**Data / specification**

1. **Flow was silently dropped from every model.** The registry hard-coded
   `"Flow_pcu/hr"`, but the column in `data3.xlsx` is `"Flow_pcu/hr/m"`. The line
   `COVARIATES = {k: v for k, v in REGISTRY.items() if v["col"] in df.columns}` then
   removed it without warning — while Cell 4 used the correct name, so the Step-1
   table showed flow but no model ever saw it. *Fixed:* columns are resolved by
   pattern matching and every registry entry is asserted to exist.
2. **`Site` was dropped on a justification the data contradict.** The markdown claimed
   the corridors carry ~3000 vs ~6000 pcu/hr so `Site` is collinear with flow. In
   `data3` the two corridors average ≈485 and ≈495 pcu/hr/m; Spearman(flow, site) ≈ 0.03
   and VIF(site) ≈ 1.9. `Site` is in fact correlated with *target speed* (ρ ≈ −0.53),
   not flow. *Fixed:* nothing is dropped; the claim must be removed from the manuscript.
3. **Flow's reporting unit was wrong** — `per +1000 pcu/hr` on a variable ranging
   197–922 pcu/hr/m. *Fixed:* per +100 pcu/hr/m.
4. **No missing-data or non-positive-response handling.** *Fixed:* explicit filtering,
   plus **joint listwise deletion** so the null and covariate models are fitted on
   identical rows (see 9 below).
5. **Hard-coded `DROP_PAIRS`.** *Fixed:* rule-based on `MIN_N_FIT`, with an override hook.

**Distribution fitting**

6. **Selection and estimation used different models.** Step 3 fitted with a *free*
   location; Steps 5+ fitted with `loc = 0`. The "best" distribution was therefore
   chosen under a parameterisation the AFT never used. Free `loc` also inflates the
   likelihood of 3-parameter families (the threshold drifts to `min(t)`, an unbounded-
   likelihood problem), so the AIC comparison against 2-parameter families was unfair.
   *Fixed:* selection uses `loc = 0` by default (`FIT_FLOC0`); free-`loc` fits are kept
   as a sensitivity sheet.
7. **`k = len(params)` counted fixed parameters as free**, biasing AIC/BIC once `loc`
   is constrained. *Fixed:* `k = numargs + 1 (+1 if loc free)`.
8. **The reported KS p-value is invalid.** `stats.kstest` assumes the parameters are
   known, not estimated from the same sample; it is strongly anti-conservative here,
   and the response is discretised to 1/30 s so ties compound the problem. *Fixed:*
   a parametric-bootstrap KS p-value for the selected distribution, plus an
   Anderson–Darling statistic (more sensitive in the tails, which is what matters for
   headway). Also added: ΔAIC, Akaike weights, BIC, AICc, and an explicit
   *selection-certainty* flag when the top two families are within 4 AIC.
9. **`fits[0]` crashes if every candidate fails.** *Fixed:* guarded.

**AFT estimation**

10. **The likelihood-ratio test could be invalid without any warning.** Rows with a
    missing covariate produced `NaN` in `z`, the likelihood collapsed to the `1e12`
    penalty, and the optimiser still returned numbers that were written to the table.
    Separately, `LR = 2*(-r1.fun - ll0)` can come out **negative** when the optimiser
    finds a worse covariate fit than the null — `chi2.sf` then quietly returns ≈1.
    *Fixed:* joint listwise deletion, null refitted on the same rows, warm start from
    the null so the covariate model can never do worse, LR clamped at 0, and any such
    event flagged in a `warning` column.
11. **Numerical overflow risk.** With `flow` restored, `exp(b0 + b1·z)` on centred but
    unscaled values of ±200 overflows. *Fixed:* continuous covariates are
    z-standardised inside the optimiser and coefficients are back-transformed for
    reporting.
12. **No standard errors, no confidence intervals, no Wald tests.** A TRB reviewer
    will reject point estimates with no uncertainty. *Fixed:* SEs from a numerical
    observed-information matrix; Wald *z*, *p*, and 95 % CIs on the % change scale.
13. **No multiplicity control in the screen** (10 pairs × 6 covariates = 60 tests).
    *Fixed:* Benjamini–Hochberg within pair and across the whole screen.
14. **Single-start Nelder–Mead/Powell, and `converged` could report the flag of a
    result that was not the one retained.** *Fixed:* multi-start with a chained polish
    and honest convergence reporting.
15. **No multivariable model existed.** v3 screened covariates one at a time and then
    listed the winners as the "final set" — but never re-estimated them jointly, so no
    coefficient was ever adjusted for the others. *Fixed:* Cell 12 fits the compound
    model (null vs selected vs full), with a joint LR test, AIC/BIC comparison and
    McFadden $R^2$.

**Statistics elsewhere**

16. **Rank-biserial sign was reversed** (`1 − 2U/n_1n_2` with scipy's `U` for the first
    sample), so the direction of every binary effect in Table 1 was backwards.
    *Fixed*, and an explicit `Direction` column now states which level is larger.
17. **`bh_adjust` broke on `NaN` p-values.** *Fixed.*
18. **The greedy VIF filter in Step 6** dropped covariates in ΔAIC order — an unstated,
    order-dependent selection step. *Removed*, per instruction.
19. **No figures and no provenance record.** *Fixed:* Cells 13–14.

**Still open — decide before submission**

- **Right truncation.** Headways are bounded at 4.97 s with the density still rising at
  the boundary, which looks like a 5 s protocol cut-off rather than a natural bound. If
  so, every unbounded fit overstates the upper tail. Set `TRUNC = (0.0, 5.0)` in Cell 1
  to fit truncated likelihoods and report it as a sensitivity analysis.
- **Discretisation.** The response is an exact multiple of 1/30 s (30 fps video).
  Continuous-density fitting is an approximation; state it as a limitation.
- **Pair-level sample sizes.** With `MIN_N_FIT = 30`, `PR_following_MT_2W` (n = 23) and
  `PR_following_NMT_2W` (n = 10) are dropped. Multi-covariate models on n ≈ 40 pairs
  (`PR_following_4W`, `BTW_following_NMT_2W`) remain thin — report their CIs, do not
  over-interpret.
- **Independence.** Consecutive headways in the same platoon/site are unlikely to be
  independent. A pair-level random effect or cluster-robust SEs would be the stronger
  specification; at minimum, acknowledge it.


In [1]:
# =============================================================================
# Cell 1 — Imports, paths, global configuration
# =============================================================================
import os, sys, json, warnings, itertools, platform
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy import stats, optimize

warnings.simplefilter("ignore")

# ---------------------------------------------------------------- paths ------
BASE      = r"D:\Headway"
DATA_PATH = os.path.join(BASE, "data3.xlsx")
TABLES    = os.path.join(BASE, "Tables")
GRAPHICS  = os.path.join(BASE, "Graphics")
os.makedirs(TABLES,   exist_ok=True)
os.makedirs(GRAPHICS, exist_ok=True)

# ------------------------------------------------------- model definition ----
OUTCOME       = "Time_Headway"
STRATUM       = "Pair"          # models are fitted separately within each Pair

# Everything EXCEPT these enters the covariate set (per study design).
#   - OUTCOME                 : the response
#   - V_Target / V_Leading_*  : define the Pair stratum, so they cannot also be covariates
#   - Leading_Speed_km/hr     : EXACTLY collinear with Target_Speed + Speed_Difference
#                               (Speed_Difference = Target_Speed - Leading_Speed), so
#                               including it would make the design matrix singular.
EXCLUDE_FROM_COVARIATES = [OUTCOME, "V_Target", "V_Subject", "V_Leading_Class",
                           "Pair", "Leading_Speed_km/hr"]

# ----------------------------------------------------------- thresholds ------
ALPHA         = 0.05
DAIC_MIN      = 2.0     # AIC_null - AIC_cov must exceed this for "strict" improvement
MIN_BIN_GROUP = 5       # min obs in each level of a binary covariate
MIN_N_FIT     = 30      # min obs for a pair to be modelled
VIF_REPORT_HI = 10.0    # VIF label thresholds -- REPORTING ONLY, nothing is dropped
VIF_REPORT_MO = 5.0

# ------------------------------------------------- distribution candidates ---
CANDIDATE_DISTS = ["lognorm", "gamma", "weibull_min", "invgauss", "expon",
                   "fisk", "pearson3", "gengamma", "rayleigh"]
FLAG_DISTS = set()      # extreme-value families to flag; genextreme is not a candidate

# Marginal fits are run with loc fixed at 0 so that Step 3 selection and the
# Step 5/7 AFT models use the SAME parameterisation. Free-loc fits are still
# produced as a sensitivity sheet.
FIT_FLOC0     = True

# Optional truncation of the likelihood. data3 headways are bounded above at
# 5 s by the extraction protocol; set TRUNC = (0.0, 5.0) to fit truncated
# likelihoods, or None to ignore truncation (matches the original pipeline).
TRUNC         = None            # e.g. (0.0, 5.0)

# Parametric-bootstrap KS p-value (the analytic KS p is invalid when parameters
# are estimated from the same sample). Applied to the best distribution only.
N_BOOT_KS     = 200             # set to 0 to skip
RNG_SEED      = 20250101
rng = np.random.default_rng(RNG_SEED)

# Numerical settings for the AFT optimiser
N_STARTS      = 3               # multi-start jitters per model
BIG_NLL       = 1e10

# ------------------------------------------------------------ formatting -----
SYMBOL = {"Spearman": "\u03c1", "Mann-Whitney U": "r", "Kruskal-Wallis": "\u03b5\u00b2"}

def magnitude(es, symbol):
    if pd.isna(es):
        return ""
    a = abs(es)
    if symbol in ("\u03c1", "r"):
        return ("negligible" if a < 0.10 else "small" if a < 0.30 else
                "medium" if a < 0.50 else "large")
    return ("negligible" if a < 0.01 else "small" if a < 0.06 else
            "medium" if a < 0.14 else "large")

def fmt_p(p):
    if pd.isna(p):
        return "n/a"
    return "<0.001" if float(p) < 1e-3 else round(float(p), 4)

def stars(p):
    if pd.isna(p):
        return ""
    p = float(p)
    return "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"

plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 300, "font.size": 9,
                     "axes.grid": True, "grid.alpha": 0.25,
                     "savefig.bbox": "tight"})

print(f"python {platform.python_version()} | numpy {np.__version__} | "
      f"scipy {stats.__name__ and __import__('scipy').__version__} | pandas {pd.__version__}")
print("Tables  ->", TABLES)
print("Graphics->", GRAPHICS)

python 3.14.6 | numpy 2.4.6 | scipy 1.18.0 | pandas 2.3.3
Tables  -> D:\Headway\Tables
Graphics-> D:\Headway\Graphics


In [2]:
# =============================================================================
# Cell 2 — Load data3, audit, and build the covariate registry AUTOMATICALLY
#          (every column that is not the outcome / stratum / leading speed)
# =============================================================================
df = pd.read_excel(DATA_PATH)
df.columns = [str(c).strip() for c in df.columns]

SUBJECT_COL = "V_Target" if "V_Target" in df.columns else "V_Subject"

def find_col(patterns, cols):
    """Resolve a column by case-insensitive substring patterns (all must match)."""
    for c in cols:
        low = c.lower()
        if all(p in low for p in patterns):
            return c
    return None

# resolved by pattern, NOT hard-coded -> immune to the pcu/hr vs pcu/hr/m mismatch
SPEED_COL   = find_col(["target", "speed"], df.columns) or find_col(["subject", "speed"], df.columns)
LEADSPD_COL = find_col(["leading", "speed"], df.columns)
FLOW_COL    = find_col(["flow"], df.columns)
SPDDIF_COL  = find_col(["speed", "differ"], df.columns)

print("Resolved columns")
print(f"  outcome        : {OUTCOME}")
print(f"  stratum        : {STRATUM}")
print(f"  subject class  : {SUBJECT_COL}")
print(f"  target speed   : {SPEED_COL}")
print(f"  leading speed  : {LEADSPD_COL}   [EXCLUDED from covariates]")
print(f"  speed diff     : {SPDDIF_COL}")
print(f"  flow           : {FLOW_COL}")

# --------------------------------------------------------------- audit -------
n0 = len(df)
df = df[np.isfinite(pd.to_numeric(df[OUTCOME], errors="coerce"))].copy()
df = df[df[OUTCOME] > 0].copy()
print(f"\nrows: {n0} -> {len(df)} after dropping non-positive / missing {OUTCOME}")
print(f"missing cells anywhere: {int(df.isna().sum().sum())}")
print(f"{OUTCOME}: min={df[OUTCOME].min():.3f}  max={df[OUTCOME].max():.3f}  "
      f"median={df[OUTCOME].median():.3f}  n_unique={df[OUTCOME].nunique()}")
if TRUNC is None and df[OUTCOME].max() < 5.05:
    print("  NOTE: the sample is bounded above at ~5 s. If that ceiling is a "
          "protocol cut-off rather than a natural bound, set TRUNC=(0.0, 5.0) in Cell 1 "
          "and re-run -- untruncated fits will overstate the upper tail.")

print("\nPair sizes:")
for p, n in df[STRATUM].value_counts().sort_values(ascending=False).items():
    tag = "ok" if n >= MIN_N_FIT else ("small" if n >= 15 else "TOO SMALL")
    print(f"   {p:24s} n={n:4d}  [{tag}]")

# ------------------------------------------- automatic covariate registry ----
def _is_binary(s):
    v = pd.Series(s).dropna().unique()
    return len(v) == 2

def _numeric_unit_label(col, s):
    """Choose a reporting unit that gives an interpretable effect size."""
    rng_ = float(np.nanmax(s) - np.nanmin(s))
    if rng_ > 200:   return 100.0, "per +100 units"
    if rng_ > 20:    return 10.0,  "per +10 units"
    if rng_ > 2:     return 1.0,   "per +1 unit"
    return 0.1, "per +0.10 unit"

REGISTRY, encoded_note = {}, []
for col in df.columns:
    if col in EXCLUDE_FROM_COVARIATES:
        continue
    s = df[col]
    if pd.api.types.is_bool_dtype(s):
        newc = f"_{col}_01"
        df[newc] = s.astype(int)
        REGISTRY[col] = dict(col=newc, type="bin", unit=1.0,
                             label="True vs False", src=col, ref="False")
    elif pd.api.types.is_numeric_dtype(s):
        unit, lab = _numeric_unit_label(col, s.values.astype(float))
        # nicer unit labels for the two known continuous covariates
        if col == SPEED_COL or col == SPDDIF_COL:
            unit, lab = 1.0, "per +1 km/h"
        elif col == FLOW_COL:
            unit, lab = 100.0, "per +100 pcu/hr/m"
        REGISTRY[col] = dict(col=col, type="cont", unit=unit, label=lab, src=col, ref="")
    else:
        levels = sorted(pd.Series(s).dropna().astype(str).unique())
        if len(levels) == 2:
            newc = f"_{col}_01"
            df[newc] = (df[col].astype(str) == levels[1]).astype(int)
            REGISTRY[col] = dict(col=newc, type="bin", unit=1.0,
                                 label=f"{levels[1]} vs {levels[0]}", src=col, ref=levels[0])
        else:
            # >2 unordered levels: keep out of the AFT scale-link (would need dummies),
            # but keep it in the Step-1 association table.
            encoded_note.append(f"{col} ({len(levels)} levels) -> association table only")
            continue

COVARIATES = dict(REGISTRY)          # nothing is ever dropped from this dict
COV_ORDER  = list(COVARIATES.keys())

print("\nCovariates entering every model stage (nothing is removed downstream):")
for k, v in COVARIATES.items():
    print(f"   {k:22s} type={v['type']:4s} col={v['col']:24s} unit={v['unit']:<6g} {v['label']}")
if encoded_note:
    print("\nNot used as AFT covariates:", "; ".join(encoded_note))
print("\nExplicitly excluded by design:", [c for c in EXCLUDE_FROM_COVARIATES if c in df.columns])

Resolved columns
  outcome        : Time_Headway
  stratum        : Pair
  subject class  : V_Target
  target speed   : Target_Speed_km/hr
  leading speed  : Leading_Speed_km/hr   [EXCLUDED from covariates]
  speed diff     : Speed_Difference
  flow           : Flow_pcu/hr/m

rows: 898 -> 898 after dropping non-positive / missing Time_Headway
missing cells anywhere: 0
Time_Headway: min=0.533  max=4.967  median=2.100  n_unique=370
  NOTE: the sample is bounded above at ~5 s. If that ceiling is a protocol cut-off rather than a natural bound, set TRUNC=(0.0, 5.0) in Cell 1 and re-run -- untruncated fits will overstate the upper tail.

Pair sizes:
   BTW_following_4W         n= 250  [ok]
   BTW_following_MT_3W      n= 186  [ok]
   BTW_following_NMT_3W     n= 119  [ok]
   PR_following_MT_3W       n= 104  [ok]
   BTW_following_MT_2W      n=  73  [ok]
   PR_following_NMT_3W      n=  49  [ok]
   PR_following_4W          n=  43  [ok]
   BTW_following_NMT_2W     n=  41  [ok]
   PR_following_MT_2

In [3]:
# =============================================================================
# Cell 3 — Association helpers (effect-size symbols, magnitude, BH-FDR)
# =============================================================================
def rank_biserial(u1, n1, n2):
    """Rank-biserial correlation from scipy's U for the FIRST sample.

    Sign convention: r > 0  =>  the first group has the LARGER outcome.
    (The previous version returned 1 - 2U/(n1*n2), which reverses this.)
    """
    return 2.0 * u1 / (n1 * n2) - 1.0

def epsilon_squared(h, n):
    """eps^2 = H / (n - 1); the standard Kruskal-Wallis effect size."""
    return h / (n - 1)

def bh_adjust(pvals):
    """Benjamini-Hochberg FDR-adjusted p-values."""
    p = np.asarray(pvals, float)
    keep = np.isfinite(p)
    adj = np.full(p.shape, np.nan)
    pk = p[keep]
    n = len(pk)
    if n == 0:
        return adj
    order = np.argsort(pk)
    ranked = np.minimum.accumulate((pk[order] * n / np.arange(1, n + 1))[::-1])[::-1]
    a = np.empty(n); a[order] = np.clip(ranked, 0, 1)
    adj[keep] = a
    return adj

def associate(data, variables):
    """var -> (Scale, Test, Symbol, EffectSize, p_raw, p_BH, n_used, direction)."""
    recs = {}
    for var, scale in variables:
        if var not in data.columns:
            continue
        sub = data[[var, OUTCOME]].dropna()
        n = len(sub)
        if n < 8:
            continue
        if scale == "cont":
            if sub[var].std() == 0:
                continue
            es, p = stats.spearmanr(sub[var], sub[OUTCOME])
            test, direction = "Spearman", ("higher -> longer" if es > 0 else "higher -> shorter")
        elif scale == "bin":
            groups = [(k, g[OUTCOME].values) for k, g in sub.groupby(var, observed=True)]
            if len(groups) != 2 or min(len(g) for _, g in groups) < 3:
                continue
            (k0, g0), (k1, g1) = groups
            u1, p = stats.mannwhitneyu(g0, g1, alternative="two-sided")
            es = rank_biserial(u1, len(g0), len(g1))
            test = "Mann-Whitney U"
            direction = f"{k0} > {k1}" if es > 0 else f"{k1} > {k0}"
        else:
            groups = [g[OUTCOME].values for _, g in sub.groupby(var, observed=True)]
            groups = [g for g in groups if len(g) >= 3]
            if len(groups) < 2:
                continue
            h, p = stats.kruskal(*groups)
            es, test, direction = epsilon_squared(h, n), "Kruskal-Wallis", "omnibus"
        recs[var] = (scale, test, SYMBOL[test], round(float(es), 4), float(p), np.nan, n, direction)

    if recs:
        padj = bh_adjust([v[4] for v in recs.values()])
        for (k, v), pa in zip(list(recs.items()), padj):
            recs[k] = v[:5] + (float(pa),) + v[6:]
    return recs

def detail_frame(rec):
    out = [dict(Variable=k, Scale=v[0], Test=v[1], Symbol=v[2], EffectSize=v[3],
                Magnitude=magnitude(v[3], v[2]), Direction=v[7], n=v[6],
                p_raw=fmt_p(v[4]), p_BH=fmt_p(v[5]), sig=stars(v[5]),
                Significant_BH="Yes" if (pd.notna(v[5]) and v[5] < ALPHA) else "No")
           for k, v in rec.items()]
    d = pd.DataFrame(out)
    if len(d):
        d = d.reindex(d["EffectSize"].abs().sort_values(ascending=False).index).reset_index(drop=True)
    return d

print("Association helpers ready (rank-biserial sign corrected; BH is NaN-safe).")

Association helpers ready (rank-biserial sign corrected; BH is NaN-safe).


In [4]:
# =============================================================================
# Cell 4 — STEP 1: Inferential statistics, Overall / BTW / PR side by side
#          -> Tables/01_inferential_stats.xlsx
# =============================================================================
def _scale_of(col):
    s = df[col]
    if pd.api.types.is_bool_dtype(s):
        return "bin"
    if not pd.api.types.is_numeric_dtype(s) or col in (STRATUM, "V_Leading_Class"):
        return "multi" if s.nunique() > 2 else "bin"
    return "cont"

# every variable of interest, including the two we exclude from the AFT models
assoc_vars = []
for col in [STRATUM, "V_Leading_Class", SPEED_COL, LEADSPD_COL, SPDDIF_COL, FLOW_COL]:
    if col and col in df.columns:
        assoc_vars.append((col, _scale_of(col)))
for k, v in COVARIATES.items():
    if not any(k == a for a, _ in assoc_vars):
        assoc_vars.append((k, "bin" if v["type"] == "bin" else "cont"))
assoc_vars = list(dict.fromkeys(assoc_vars))

overall_vars = [(SUBJECT_COL, "bin")] + assoc_vars
group_vars   = assoc_vars

rec_all = associate(df, overall_vars)
rec_btw = associate(df[df[SUBJECT_COL] == "BTW"], group_vars)
rec_pr  = associate(df[df[SUBJECT_COL] == "PR"],  group_vars)

def _cell(rec, var):
    if var not in rec:
        return (np.nan, "", np.nan)
    scale, test, sym, es, p_raw, p_bh, n, direc = rec[var]
    return (es, magnitude(es, sym), p_bh)

rows = []
for var, _ in overall_vars:
    if var in rec_all:
        scale, test, sym = rec_all[var][0], rec_all[var][1], rec_all[var][2]
    else:
        scale = test = sym = ""
    esO, magO, pO = _cell(rec_all, var)
    esB, magB, pB = _cell(rec_btw, var)
    esP, magP, pP = _cell(rec_pr,  var)
    rows.append(dict(Variable=var, Scale=scale, Test=test, Symbol=sym,
                     ES_Overall=esO, Mag_Overall=magO, p_Overall=fmt_p(pO), sig_Overall=stars(pO),
                     ES_BTW=esB, Mag_BTW=magB, p_BTW=fmt_p(pB), sig_BTW=stars(pB),
                     ES_PR=esP, Mag_PR=magP, p_PR=fmt_p(pP), sig_PR=stars(pP)))
inferential = pd.DataFrame(rows)

# descriptive summary of the outcome by pair (useful Table 1 material)
desc = (df.groupby(STRATUM)[OUTCOME]
          .agg(N="size", Mean="mean", SD="std", Min="min", Q1=lambda s: s.quantile(.25),
               Median="median", Q3=lambda s: s.quantile(.75), Max="max",
               CV=lambda s: s.std() / s.mean(),
               Skew=lambda s: stats.skew(s), Kurtosis=lambda s: stats.kurtosis(s))
          .round(4).sort_values("N", ascending=False).reset_index())

xl1 = os.path.join(TABLES, "01_inferential_stats.xlsx")
with pd.ExcelWriter(xl1, engine="openpyxl") as xl:
    inferential.to_excel(xl,       sheet_name="Summary_PR_BTW", index=False)
    desc.to_excel(xl,              sheet_name="Descriptives_by_pair", index=False)
    detail_frame(rec_all).to_excel(xl, sheet_name="Overall_detail", index=False)
    detail_frame(rec_btw).to_excel(xl, sheet_name="BTW_detail", index=False)
    detail_frame(rec_pr).to_excel(xl,  sheet_name="PR_detail", index=False)
print("Saved Excel 1:", xl1)
inferential

Saved Excel 1: D:\Headway\Tables\01_inferential_stats.xlsx


,Variable,Scale,Test,Symbol,ES_Overall,Mag_Overall,p_Overall,sig_Overall,ES_BTW,Mag_BTW,p_BTW,sig_BTW,ES_PR,Mag_PR,p_PR,sig_PR
0,V_Target,bin,Mann-Whitney U,r,-0.4745,medium,<0.001,***,NaN,,n/a,,NaN,,n/a,
1,Pair,multi,Kruskal-Wallis,ε²,0.1775,large,<0.001,***,0.0706,medium,<0.001,***,0.0130,small,0.562,ns
2,V_Leading_Class,multi,Kruskal-Wallis,ε²,0.0248,small,<0.001,***,0.0706,medium,<0.001,***,0.0130,small,0.562,ns
3,Target_Speed_km/hr,cont,Spearman,ρ,-0.3901,medium,<0.001,***,-0.3126,medium,<0.001,***,-0.2505,small,0.0011,**
4,Leading_Speed_km/hr,cont,Spearman,ρ,-0.0194,negligible,0.5624,ns,-0.0107,negligible,0.783,ns,0.0724,negligible,0.4738,ns
5,Speed_Difference,cont,Spearman,ρ,-0.3404,medium,<0.001,***,-0.3126,medium,<0.001,***,-0.1917,small,0.0161,*
6,Flow_pcu/hr/m,cont,Spearman,ρ,0.0733,negligible,0.0313,*,0.0560,negligible,0.1902,ns,0.0850,negligible,0.4494,ns
7,Off_centeredness,bin,Mann-Whitney U,r,0.1229,small,0.0037,**,0.1347,small,0.0079,**,0.0808,negligible,0.4738,ns
8,Occupancy,bin,Mann-Whitney U,r,0.1697,small,<0.001,***,0.0545,negligible,0.4258,ns,0.1854,small,0.0749,ns
9,Site,bin,Mann-Whitney U,r,-0.1665,small,<0.001,***,-0.2431,small,<0.001,***,-0.0530,negligible,0.562,ns


In [5]:
# =============================================================================
# Cell 5 — STEP 2: Multicollinearity diagnostics.  REPORT ONLY.
#          No covariate is removed here or anywhere downstream.
#          -> Tables/02_vif_multicollinearity.xlsx  +  Graphics/fig02_*.png
# =============================================================================
def compute_vif(frame, cols, names=None):
    """VIF_j = 1 / (1 - R2_j) obtained from the inverse correlation matrix.

    Returns a tidy DataFrame. Degenerate columns (constant, or fewer rows than
    columns) are reported as such instead of silently disappearing.
    """
    names = names or cols
    X = frame[cols].apply(pd.to_numeric, errors="coerce").dropna()
    out = []
    const = [c for c in cols if X[c].nunique() <= 1] if len(X) else list(cols)
    usable = [c for c in cols if c not in const]
    vif_map = {}
    if len(usable) >= 2 and len(X) > len(usable) + 2:
        R = np.corrcoef(X[usable].values.T)
        try:
            iR = np.linalg.inv(R)
        except np.linalg.LinAlgError:
            iR = np.linalg.pinv(R)
        vif_map = {c: float(iR[i, i]) for i, c in enumerate(usable)}
    for c, nm in zip(cols, names):
        v = vif_map.get(c, np.nan)
        out.append(dict(Covariate=nm, Column=c, n_used=int(len(X)),
                        VIF=round(v, 3) if np.isfinite(v) else np.nan,
                        Tolerance=round(1 / v, 3) if np.isfinite(v) and v > 0 else np.nan,
                        R2_on_others=round(1 - 1 / v, 3) if np.isfinite(v) and v > 0 else np.nan,
                        Status=("constant / not estimable" if c in const else
                                "high (>10)" if v > VIF_REPORT_HI else
                                "moderate (>5)" if v > VIF_REPORT_MO else "ok")))
    return pd.DataFrame(out)

vif_cols  = [v["col"] for v in COVARIATES.values()]
vif_names = list(COVARIATES.keys())

# ---- global VIF across the pooled sample ------------------------------------
vif_global = compute_vif(df, vif_cols, vif_names).sort_values("VIF", ascending=False).reset_index(drop=True)

# ---- per-pair VIF: this is the one that matters, models are fitted per pair --
per_pair = []
for pair, g in df.groupby(STRATUM):
    t = compute_vif(g, vif_cols, vif_names)
    t.insert(0, "Pair", pair); t.insert(1, "N", len(g))
    per_pair.append(t)
vif_by_pair = pd.concat(per_pair, ignore_index=True)
vif_pair_matrix = (vif_by_pair.pivot(index="Pair", columns="Covariate", values="VIF")
                              .reindex(columns=vif_names).round(2))

# ---- correlation structure (Spearman, all covariates) -----------------------
corr_all = df[vif_cols].apply(pd.to_numeric, errors="coerce").corr(method="spearman").round(3)
corr_all.index = corr_all.columns = vif_names

# ---- design-matrix conditioning ---------------------------------------------
Xs = df[vif_cols].apply(pd.to_numeric, errors="coerce").dropna().values.astype(float)
Xs = (Xs - Xs.mean(0)) / np.where(Xs.std(0) == 0, 1, Xs.std(0))
cond_number = float(np.linalg.cond(np.column_stack([np.ones(len(Xs)), Xs])))
cond_tbl = pd.DataFrame([dict(Metric="Condition number (scaled design matrix)",
                              Value=round(cond_number, 2),
                              Interpretation=("severe (>30)" if cond_number > 30 else
                                              "moderate (>15)" if cond_number > 15 else "ok"))])

xl2 = os.path.join(TABLES, "02_vif_multicollinearity.xlsx")
with pd.ExcelWriter(xl2, engine="openpyxl") as xl:
    vif_global.to_excel(xl,      sheet_name="VIF_global", index=False)
    vif_pair_matrix.to_excel(xl, sheet_name="VIF_by_pair_matrix")
    vif_by_pair.to_excel(xl,     sheet_name="VIF_by_pair_long", index=False)
    corr_all.to_excel(xl,        sheet_name="Spearman_corr")
    cond_tbl.to_excel(xl,        sheet_name="Conditioning", index=False)
print("Saved Excel 2:", xl2)

# ---- graphics ---------------------------------------------------------------
fig, ax = plt.subplots(figsize=(6.4, 3.4))
vg = vif_global.dropna(subset=["VIF"]).sort_values("VIF")
ax.barh(vg["Covariate"], vg["VIF"], color="#4C72B0")
ax.axvline(VIF_REPORT_MO, ls="--", c="#DD8452", lw=1, label=f"VIF={VIF_REPORT_MO:g}")
ax.axvline(VIF_REPORT_HI, ls="--", c="#C44E52", lw=1, label=f"VIF={VIF_REPORT_HI:g}")
ax.set_xlabel("Variance inflation factor"); ax.set_title("Global VIF (pooled sample)")
ax.legend(frameon=False, fontsize=8)
fig.savefig(os.path.join(GRAPHICS, "fig02a_vif_global.png")); plt.close(fig)

fig, ax = plt.subplots(figsize=(5.8, 5.0))
im = ax.imshow(corr_all.values, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(vif_names))); ax.set_xticklabels(vif_names, rotation=45, ha="right")
ax.set_yticks(range(len(vif_names))); ax.set_yticklabels(vif_names)
for i in range(len(vif_names)):
    for j in range(len(vif_names)):
        ax.text(j, i, f"{corr_all.values[i, j]:.2f}", ha="center", va="center", fontsize=7)
ax.set_title("Spearman correlation among covariates"); ax.grid(False)
fig.colorbar(im, ax=ax, shrink=0.8)
fig.savefig(os.path.join(GRAPHICS, "fig02b_corr_covariates.png")); plt.close(fig)

print(f"Condition number = {cond_number:.2f}")
print("Max global VIF   = "
      f"{np.nanmax(vif_global['VIF'].values):.2f} ({vif_global.iloc[0]['Covariate']})")
print("REPORT ONLY: no covariate is dropped. All", len(COVARIATES),
      "covariates continue to Steps 5-7:", list(COVARIATES.keys()))
vif_global

Saved Excel 2: D:\Headway\Tables\02_vif_multicollinearity.xlsx
Condition number = 2.41
Max global VIF   = 1.93 (Target_Speed_km/hr)
REPORT ONLY: no covariate is dropped. All 6 covariates continue to Steps 5-7: ['Target_Speed_km/hr', 'Speed_Difference', 'Off_centeredness', 'Occupancy', 'Flow_pcu/hr/m', 'Site']


,Covariate,Column,n_used,VIF,Tolerance,R2_on_others,Status
0,Target_Speed_km/hr,Target_Speed_km/hr,898,1.931,0.518,0.482,ok
1,Site,_Site_01,898,1.868,0.535,0.465,ok
2,Speed_Difference,Speed_Difference,898,1.298,0.770,0.230,ok
3,Occupancy,_Occupancy_01,898,1.070,0.934,0.066,ok
4,Flow_pcu/hr/m,Flow_pcu/hr/m,898,1.023,0.978,0.022,ok
5,Off_centeredness,_Off_centeredness_01,898,1.019,0.981,0.019,ok


In [6]:
# =============================================================================
# Cell 6 — Marginal distribution machinery (Step 3)
#   * loc fixed at 0 by default so that the SELECTED distribution and the
#     Step 5/7 AFT model share one parameterisation
#   * k counts FREE parameters only (loc is not free when FIT_FLOC0)
#   * optional truncated likelihood
#   * Anderson-Darling in addition to KS
#   * parametric-bootstrap KS p-value (the analytic one is invalid for
#     estimated parameters -- Lilliefors problem)
# =============================================================================
def _log_trunc_const(dist, params, trunc):
    """log[F(U) - F(L)] for the truncated likelihood; 0.0 when TRUNC is None."""
    if trunc is None:
        return 0.0
    lo, hi = trunc
    m = dist.cdf(hi, *params) - dist.cdf(lo, *params)
    return np.log(m) if m > 0 else -np.inf

def _safe_fit(dist, data, floc0):
    try:
        return dist.fit(data, floc=0) if floc0 else dist.fit(data)
    except Exception:
        try:
            return dist.fit(data)
        except Exception:
            return None

def anderson_darling(data, dist, params):
    x = np.sort(np.asarray(data, float))
    n = len(x)
    u = np.clip(dist.cdf(x, *params), 1e-12, 1 - 1e-12)
    i = np.arange(1, n + 1)
    return float(-n - np.sum((2 * i - 1) * (np.log(u) + np.log(1 - u[::-1]))) / n)

def fit_marginal(name, data, floc0=None, trunc=None):
    """Fit one candidate distribution; return a dict or None on failure."""
    floc0 = FIT_FLOC0 if floc0 is None else floc0
    dist = getattr(stats, name)
    params = _safe_fit(dist, data, floc0)
    if params is None:
        return None
    try:
        lp = dist.logpdf(data, *params)
        if not np.all(np.isfinite(lp)):
            return None
        ll = float(np.sum(lp) - len(data) * _log_trunc_const(dist, params, trunc))
        if not np.isfinite(ll):
            return None
        n = len(data)
        # FREE parameters: shapes + scale (+ loc only when it was estimated)
        k = dist.numargs + 1 + (0 if floc0 else 1)
        ks, ks_p_analytic = stats.kstest(data, name, args=params)
        ad = anderson_darling(data, dist, params)
        labels = ([s.strip() for s in dist.shapes.split(",")] if dist.shapes else []) + ["loc", "scale"]
        pstr = ", ".join(f"{l}={v:.4f}" for l, v in zip(labels, params))
        return dict(Distribution=name, k=k, n=n, LogLik=ll,
                    AIC=2 * k - 2 * ll, BIC=k * np.log(n) - 2 * ll,
                    AICc=2 * k - 2 * ll + (2 * k * (k + 1) / (n - k - 1) if n - k - 1 > 0 else np.nan),
                    KS=float(ks), KS_p_analytic=float(ks_p_analytic), AD=ad,
                    Params=pstr, params=params, loc_fixed=bool(floc0))
    except Exception:
        return None

def bootstrap_ks_p(name, params, n, ks_obs, n_boot, floc0=None, seed=None):
    """Parametric-bootstrap p-value for the KS statistic with estimated params."""
    if not n_boot:
        return np.nan
    floc0 = FIT_FLOC0 if floc0 is None else floc0
    dist = getattr(stats, name)
    r = np.random.default_rng(RNG_SEED if seed is None else seed)
    cnt, done = 0, 0
    for _ in range(n_boot):
        try:
            xb = dist.rvs(*params, size=n, random_state=r)
            xb = xb[np.isfinite(xb) & (xb > 0)]
            if len(xb) < max(10, n // 2):
                continue
            pb = _safe_fit(dist, xb, floc0)
            if pb is None:
                continue
            ksb, _ = stats.kstest(xb, name, args=pb)
            cnt += int(ksb >= ks_obs); done += 1
        except Exception:
            continue
    return round((cnt + 1) / (done + 1), 4) if done else np.nan

def akaike_weights(aics):
    a = np.asarray(aics, float)
    d = a - np.nanmin(a)
    w = np.exp(-0.5 * d)
    return d, w / np.nansum(w)

print(f"Marginal fitter ready | loc fixed at 0: {FIT_FLOC0} | truncation: {TRUNC} | "
      f"bootstrap KS reps: {N_BOOT_KS}")

Marginal fitter ready | loc fixed at 0: True | truncation: None | bootstrap KS reps: 200


In [7]:
# =============================================================================
# Cell 7 — STEP 3: Fit every candidate distribution per pair, report all
#          -> Tables/03_distribution_fits.xlsx
# =============================================================================
all_rows, best_rows, pair_best, pair_best_params, freeloc_rows = [], [], {}, {}, []

for pair, g in df.groupby(STRATUM):
    t = g[OUTCOME].dropna().values.astype(float)
    n = len(t)

    fits = [f for f in (fit_marginal(nm, t, floc0=FIT_FLOC0, trunc=TRUNC)
                        for nm in CANDIDATE_DISTS) if f]
    if not fits:                                        # guard: the old code did fits[0]
        print(f"  !! {pair}: NO candidate distribution converged -- skipped")
        continue
    fits.sort(key=lambda d: d["AIC"])
    dA, wA = akaike_weights([f["AIC"] for f in fits])

    for rank, (f, d_, w_) in enumerate(zip(fits, dA, wA), 1):
        all_rows.append(dict(Pair=pair, N=n, Rank=rank, Distribution=f["Distribution"],
                             k=f["k"], LogLik=round(f["LogLik"], 3),
                             AIC=round(f["AIC"], 2), dAIC=round(d_, 2),
                             Akaike_weight=round(w_, 4),
                             BIC=round(f["BIC"], 2), AICc=round(f["AICc"], 2),
                             KS=round(f["KS"], 4), KS_p_analytic=round(f["KS_p_analytic"], 4),
                             AD=round(f["AD"], 3), Params=f["Params"]))

    b = fits[0]
    pair_best[pair] = b["Distribution"]
    pair_best_params[pair] = b["params"]
    runner_up_gap = float(dA[1]) if len(dA) > 1 else np.inf
    ks_p_boot = bootstrap_ks_p(b["Distribution"], b["params"], n, b["KS"], N_BOOT_KS,
                               floc0=FIT_FLOC0, seed=RNG_SEED + abs(hash(pair)) % 10000)
    best_rows.append(dict(
        Pair=pair, N=n, Best_Distribution=b["Distribution"], k=b["k"],
        AIC=round(b["AIC"], 2), BIC=round(b["BIC"], 2),
        Runner_up=fits[1]["Distribution"] if len(fits) > 1 else "-",
        dAIC_to_runner_up=round(runner_up_gap, 2),
        Akaike_weight=round(float(wA[0]), 4),
        selection_certainty=("decisive" if runner_up_gap >= 10 else
                             "clear" if runner_up_gap >= 4 else
                             "weak (top models indistinguishable)"),
        KS=round(b["KS"], 4), KS_p_analytic=round(b["KS_p_analytic"], 4),
        KS_p_bootstrap=ks_p_boot, AD=round(b["AD"], 3),
        GOF_verdict=("adequate" if (ks_p_boot if pd.notna(ks_p_boot) else b["KS_p_analytic"]) > ALPHA
                     else "rejected at 5%"),
        extreme_value_flag="Yes" if b["Distribution"] in FLAG_DISTS else "No",
        fit_flag="ok" if n >= MIN_N_FIT else ("small" if n >= 15 else "TOO SMALL"),
        Params=b["Params"]))

    # sensitivity: same candidates with loc estimated freely
    for f in (fit_marginal(nm, t, floc0=False, trunc=TRUNC) for nm in CANDIDATE_DISTS):
        if f:
            freeloc_rows.append(dict(Pair=pair, N=n, Distribution=f["Distribution"], k=f["k"],
                                     AIC=round(f["AIC"], 2), BIC=round(f["BIC"], 2),
                                     KS=round(f["KS"], 4), Params=f["Params"]))

all_fits           = pd.DataFrame(all_rows)
best_per_pair_dist = pd.DataFrame(best_rows).sort_values("N", ascending=False).reset_index(drop=True)
freeloc_fits       = pd.DataFrame(freeloc_rows)
if len(freeloc_fits):
    freeloc_fits["Rank"] = freeloc_fits.groupby("Pair")["AIC"].rank(method="first").astype(int)
    freeloc_fits = freeloc_fits.sort_values(["Pair", "Rank"])

# how often does each family win / how good is it on average?
fam_summary = (all_fits.groupby("Distribution")
                       .agg(times_best=("Rank", lambda s: int((s == 1).sum())),
                            mean_rank=("Rank", "mean"),
                            mean_dAIC=("dAIC", "mean"),
                            mean_Akaike_w=("Akaike_weight", "mean"))
                       .round(3).sort_values(["times_best", "mean_rank"], ascending=[False, True])
                       .reset_index())

xl3 = os.path.join(TABLES, "03_distribution_fits.xlsx")
with pd.ExcelWriter(xl3, engine="openpyxl") as xl:
    best_per_pair_dist.to_excel(xl, sheet_name="Best_per_pair", index=False)
    all_fits.to_excel(xl,           sheet_name="All_fits_loc0", index=False)
    fam_summary.to_excel(xl,        sheet_name="Family_summary", index=False)
    if len(freeloc_fits):
        freeloc_fits.to_excel(xl,   sheet_name="Sensitivity_free_loc", index=False)
print("Saved Excel 3:", xl3)

weak = best_per_pair_dist.loc[best_per_pair_dist["selection_certainty"].str.startswith("weak"), "Pair"].tolist()
if weak:
    print("Selection is NOT decisive (top models within 4 AIC) for:", weak)
    print("  -> report these as 'best among near-equivalent candidates', not as 'the' distribution.")
best_per_pair_dist

Saved Excel 3: D:\Headway\Tables\03_distribution_fits.xlsx
Selection is NOT decisive (top models within 4 AIC) for: ['BTW_following_4W', 'BTW_following_MT_3W', 'BTW_following_NMT_3W', 'PR_following_MT_3W', 'BTW_following_MT_2W', 'PR_following_NMT_3W', 'PR_following_4W', 'BTW_following_NMT_2W', 'PR_following_MT_2W']
  -> report these as 'best among near-equivalent candidates', not as 'the' distribution.


,Pair,N,Best_Distribution,k,AIC,BIC,Runner_up,dAIC_to_runner_up,Akaike_weight,selection_certainty,KS,KS_p_analytic,KS_p_bootstrap,AD,GOF_verdict,extreme_value_flag,fit_flag,Params
0,BTW_following_4W,250,gengamma,3,623.62,634.19,weibull_min,0.90,0.4961,weak (top models indistinguishable),0.0302,0.9715,0.7711,0.210,adequate,No,ok,"a=2.0043, c=1.9492, loc=0.0000, scale=1.7009"
1,BTW_following_MT_3W,186,gamma,2,442.49,448.94,lognorm,0.86,0.3467,weak (top models indistinguishable),0.0486,0.7527,0.3682,0.336,adequate,No,ok,"a=4.8580, loc=0.0000, scale=0.3835"
2,BTW_following_NMT_3W,119,gamma,2,298.01,303.57,invgauss,0.04,0.3201,weak (top models indistinguishable),0.0663,0.6466,0.2338,0.720,adequate,No,ok,"a=4.5347, loc=0.0000, scale=0.4224"
3,PR_following_MT_3W,104,weibull_min,2,278.65,283.93,gengamma,1.43,0.6660,weak (top models indistinguishable),0.0760,0.5594,0.1244,0.441,adequate,No,ok,"c=3.5179, loc=0.0000, scale=3.1792"
4,BTW_following_MT_2W,73,invgauss,2,168.17,172.75,gamma,0.10,0.2936,weak (top models indistinguishable),0.0760,0.7641,0.4776,0.513,adequate,No,ok,"mu=0.2453, loc=0.0000, scale=7.1948"
5,PR_following_NMT_3W,49,weibull_min,2,140.84,144.62,gengamma,0.52,0.5347,weak (top models indistinguishable),0.0912,0.7757,0.3632,0.405,adequate,No,ok,"c=3.0628, loc=0.0000, scale=3.0337"
6,PR_following_4W,43,weibull_min,2,130.42,133.94,gengamma,0.01,0.4625,weak (top models indistinguishable),0.1427,0.3151,0.0498,0.797,rejected at 5%,No,ok,"c=3.1565, loc=0.0000, scale=3.3365"
7,BTW_following_NMT_2W,41,invgauss,2,94.17,97.60,lognorm,0.31,0.2500,weak (top models indistinguishable),0.0691,0.9819,0.9502,0.246,adequate,No,ok,"mu=0.2563, loc=0.0000, scale=6.6192"
8,PR_following_MT_2W,23,weibull_min,2,70.73,73.00,gamma,0.67,0.3041,weak (top models indistinguishable),0.1108,0.9112,0.6219,0.245,adequate,No,small,"c=3.3261, loc=0.0000, scale=3.4916"
9,PR_following_NMT_2W,10,pearson3,2,11.41,12.01,rayleigh,24.32,1.0000,decisive,0.3373,0.1621,1.0000,2.370,adequate,No,TOO SMALL,"skew=-2.9965, loc=0.0000, scale=7.2914"


In [8]:
# =============================================================================
# Cell 8 — STEP 4: Retain pairs with enough data (rule-based, with manual override)
# =============================================================================
# Rule: keep pairs with N >= MIN_N_FIT. Add names to FORCE_DROP / FORCE_KEEP to override.
FORCE_DROP, FORCE_KEEP = [], []

sizes = df[STRATUM].value_counts()
auto_drop = [p for p in sizes.index if sizes[p] < MIN_N_FIT]
DROP_PAIRS = sorted((set(auto_drop) | set(FORCE_DROP)) - set(FORCE_KEEP))

KEPT_PAIRS = (best_per_pair_dist[~best_per_pair_dist[STRATUM].isin(DROP_PAIRS)]
              .sort_values("N", ascending=False)[STRATUM].tolist())
dfk = df[df[STRATUM].isin(KEPT_PAIRS)].copy()

retention = pd.DataFrame([
    dict(Pair=p, N=int(sizes[p]), Best_Distribution=pair_best.get(p, "-"),
         Status="kept" if p in KEPT_PAIRS else "dropped",
         Reason=("N >= %d" % MIN_N_FIT) if p in KEPT_PAIRS
                else ("N < %d" % MIN_N_FIT if p in auto_drop else "manual"))
    for p in sizes.index]).sort_values("N", ascending=False).reset_index(drop=True)

print(f"Rule: keep pairs with N >= {MIN_N_FIT}")
print(f"Dropped ({len(DROP_PAIRS)}): {DROP_PAIRS}")
print(f"Kept    ({len(KEPT_PAIRS)}), {len(dfk)} of {len(df)} observations "
      f"({100*len(dfk)/len(df):.1f}%):")
for p in KEPT_PAIRS:
    print(f"   {p:24s} n={int(sizes[p]):4d}  best_dist={pair_best[p]}")
retention

Rule: keep pairs with N >= 30
Dropped (2): ['PR_following_MT_2W', 'PR_following_NMT_2W']
Kept    (8), 865 of 898 observations (96.3%):
   BTW_following_4W         n= 250  best_dist=gengamma
   BTW_following_MT_3W      n= 186  best_dist=gamma
   BTW_following_NMT_3W     n= 119  best_dist=gamma
   PR_following_MT_3W       n= 104  best_dist=weibull_min
   BTW_following_MT_2W      n=  73  best_dist=invgauss
   PR_following_NMT_3W      n=  49  best_dist=weibull_min
   PR_following_4W          n=  43  best_dist=weibull_min
   BTW_following_NMT_2W     n=  41  best_dist=invgauss


,Pair,N,Best_Distribution,Status,Reason
0,BTW_following_4W,250,gengamma,kept,N >= 30
1,BTW_following_MT_3W,186,gamma,kept,N >= 30
2,BTW_following_NMT_3W,119,gamma,kept,N >= 30
3,PR_following_MT_3W,104,weibull_min,kept,N >= 30
4,BTW_following_MT_2W,73,invgauss,kept,N >= 30
5,PR_following_NMT_3W,49,weibull_min,kept,N >= 30
6,PR_following_4W,43,weibull_min,kept,N >= 30
7,BTW_following_NMT_2W,41,invgauss,kept,N >= 30
8,PR_following_MT_2W,23,weibull_min,dropped,N < 30
9,PR_following_NMT_2W,10,pearson3,dropped,N < 30


In [9]:
# =============================================================================
# Cell 9 — Accelerated-failure-time machinery
#   log(scale_i) = b0 + b' z_i ,  shape shared across observations,  loc = 0
#
# Rewritten from v3. Fixes:
#   (a) rows with a missing covariate are dropped JOINTLY with the response, and
#       the null model is refitted on the SAME rows -> the LR test is now valid
#   (b) continuous covariates are z-standardised INSIDE the optimiser (raw flow
#       values of ~500 made exp(b0 + b1*z) overflow) and back-transformed for reporting
#   (c) multi-start optimisation + warm start from the null; the log-likelihood of
#       the covariate model can no longer come out BELOW the null undetected
#   (d) standard errors, Wald z, and 95% CI from a numerical observed-information
#       matrix -- v3 reported point effects with no uncertainty at all
#   (e) supports a multivariable (compound) fit, not just one covariate at a time
#   (f) optional truncated likelihood, consistent with Cell 6
# =============================================================================
def _eta(p, Z, ns):
    b0 = p[ns]
    if Z is None or Z.shape[1] == 0:
        return np.full(Z.shape[0] if Z is not None else 0, b0)
    return b0 + Z @ p[ns + 1:]

def _nll(p, t, Z, dist, ns, trunc):
    eta = np.clip(_eta(p, Z, ns), -25.0, 25.0)
    scale = np.exp(eta)
    shapes = p[:ns]
    with np.errstate(all="ignore"):
        lp = dist.logpdf(t, *shapes, loc=0.0, scale=scale)
        if trunc is not None:
            lo, hi = trunc
            m = (dist.cdf(hi, *shapes, loc=0.0, scale=scale) -
                 dist.cdf(lo, *shapes, loc=0.0, scale=scale))
            lp = lp - np.log(np.where(m > 0, m, np.nan))
    if not np.all(np.isfinite(lp)):
        return BIG_NLL
    v = -float(np.sum(lp))
    return v if np.isfinite(v) else BIG_NLL

def _optimise(fn, x0, args, n_starts=None, jitter=0.15):
    """Multi-start Nelder-Mead followed by a Powell polish; returns the best result."""
    n_starts = N_STARTS if n_starts is None else n_starts
    r0 = np.random.default_rng(RNG_SEED)
    best = None
    starts = [np.asarray(x0, float)]
    for _ in range(max(0, n_starts - 1)):
        starts.append(np.asarray(x0, float) + r0.normal(0, jitter, size=len(x0)))
    for s in starts:
        for meth, opt in (("Nelder-Mead", dict(maxiter=20000, maxfev=40000,
                                               xatol=1e-9, fatol=1e-9)),
                          ("Powell",      dict(maxiter=20000, maxfev=40000,
                                               xtol=1e-9, ftol=1e-9))):
            try:
                r = optimize.minimize(fn, s, args=args, method=meth, options=opt)
            except Exception:
                continue
            if np.isfinite(r.fun) and (best is None or r.fun < best.fun - 1e-12):
                best = r
            s = r.x if np.all(np.isfinite(r.x)) else s     # chain: polish from NM solution
    return best

def _hessian(fn, x, args, rel=1e-4):
    x = np.asarray(x, float); n = len(x)
    h = np.maximum(rel * np.abs(x), 1e-5)
    H = np.zeros((n, n))
    for i in range(n):
        for j in range(i, n):
            ei = np.zeros(n); ei[i] = h[i]
            ej = np.zeros(n); ej[j] = h[j]
            f1 = fn(x + ei + ej, *args); f2 = fn(x + ei - ej, *args)
            f3 = fn(x - ei + ej, *args); f4 = fn(x - ei - ej, *args)
            if max(f1, f2, f3, f4) >= BIG_NLL:
                return None
            H[i, j] = H[j, i] = (f1 - f2 - f3 + f4) / (4 * h[i] * h[j])
    return H

def _stderr(fn, x, args):
    H = _hessian(fn, x, args)
    if H is None:
        return np.full(len(x), np.nan)
    try:
        C = np.linalg.inv(H)
    except np.linalg.LinAlgError:
        C = np.linalg.pinv(H)
    d = np.diag(C)
    return np.where(d > 0, np.sqrt(np.abs(d)), np.nan)

def build_design(g, cov_names, covariates):
    """Return (t, Z, scalers, used_names, n_dropped) with joint listwise deletion."""
    cols = [covariates[c]["col"] for c in cov_names]
    sub = g[[OUTCOME] + cols].apply(pd.to_numeric, errors="coerce") if cols else \
          g[[OUTCOME]].apply(pd.to_numeric, errors="coerce")
    n_before = len(sub)
    sub = sub.dropna()
    sub = sub[sub[OUTCOME] > 0]
    t = sub[OUTCOME].values.astype(float)
    Z, scalers, used = [], {}, []
    for c in cov_names:
        meta = covariates[c]
        x = sub[meta["col"]].values.astype(float)
        if meta["type"] == "cont":
            sd = float(np.std(x, ddof=1))
            if sd <= 1e-12:
                scalers[c] = None; continue
            Z.append((x - x.mean()) / sd); scalers[c] = sd
        else:
            if len(np.unique(x)) != 2 or min((x == 0).sum(), (x == 1).sum()) < MIN_BIN_GROUP:
                scalers[c] = None; continue
            Z.append(x); scalers[c] = 1.0
        used.append(c)
    Zm = np.column_stack(Z) if Z else np.empty((len(t), 0))
    return t, Zm, scalers, used, n_before - len(sub)

def fit_aft(dist_name, t, Z, warm=None):
    """Fit log(scale) = b0 + b'z. Returns dict with params, ll, se, convergence."""
    dist = getattr(stats, dist_name)
    ns = dist.numargs
    k = Z.shape[1]
    if warm is not None and len(warm) == ns + 1 + k:
        x0 = np.asarray(warm, float)
    else:
        try:
            init = dist.fit(t, floc=0)
        except Exception:
            init = tuple([1.0] * ns) + (0.0, float(np.mean(t)))
        x0 = np.array(list(init[:ns]) + [np.log(max(init[-1], 1e-6))] + [0.0] * k)
        if warm is not None:                       # warm start from the null model
            x0[:ns + 1] = np.asarray(warm, float)[:ns + 1]
    args = (t, Z, dist, ns, TRUNC)
    r = _optimise(_nll, x0, args)
    if r is None:
        return None
    ll = -float(r.fun)
    if not np.isfinite(ll) or r.fun >= BIG_NLL:
        return None
    se = _stderr(_nll, r.x, args)
    return dict(dist=dist_name, ns=ns, k=k, x=r.x, se=se, ll=ll,
                nparam=ns + 1 + k, converged=bool(getattr(r, "success", False)),
                nll_fun=r.fun)

def effect_string(b_std, meta, sd):
    """Convert a standardised coefficient to a % change in scale per reporting unit."""
    if meta["type"] == "cont":
        b_raw = b_std / sd if (sd and np.isfinite(sd)) else np.nan
        pct = (np.exp(b_raw * meta["unit"]) - 1) * 100
        return b_raw, f"{pct:+.2f}%  {meta['label']}"
    pct = (np.exp(b_std) - 1) * 100
    return b_std, f"{pct:+.2f}%  ({meta['label']})"

print("AFT machinery ready: joint listwise deletion, internal standardisation, "
      "multi-start, numerical SEs, multivariable support.")

AFT machinery ready: joint listwise deletion, internal standardisation, multi-start, numerical SEs, multivariable support.


In [10]:
# =============================================================================
# Cell 10 — STEP 5: Univariate covariate screen inside each pair, under that
#           pair's OWN best distribution.  ALL covariates are screened.
#           -> Tables/04_covariate_screen.xlsx  +  Graphics/fig04_dAIC_heatmap.png
# =============================================================================
def screen_pair(pair, g, dist_name, covariates):
    out = []
    for cov in covariates:
        meta = covariates[cov]
        t, Z, scalers, used, ndrop = build_design(g, [cov], covariates)
        base = dict(Pair=pair, Distribution=dist_name, Covariate=cov,
                    Type=meta["type"], Unit=meta["label"], n_model=len(t),
                    n_dropped_missing=ndrop)
        if not used or len(t) < MIN_N_FIT:
            out.append({**base, "Effect": "-- insufficient variation / n --",
                        "improves": "n/a", "improves_strict": "n/a"})
            continue

        f0 = fit_aft(dist_name, t, np.empty((len(t), 0)))
        if f0 is None:
            out.append({**base, "Effect": "-- null model failed --",
                        "improves": "n/a", "improves_strict": "n/a"})
            continue
        f1 = fit_aft(dist_name, t, Z, warm=list(f0["x"]) + [0.0])
        if f1 is None:
            out.append({**base, "Effect": "-- covariate model failed --",
                        "improves": "n/a", "improves_strict": "n/a"})
            continue

        LR = 2 * (f1["ll"] - f0["ll"])
        opt_warning = "optimiser: covariate ll < null ll" if LR < -1e-6 else ""
        LR = max(LR, 0.0)
        p_lr = float(stats.chi2.sf(LR, 1))
        dAIC = LR - 2.0                                  # AIC_null - AIC_cov

        b_std = float(f1["x"][f1["ns"] + 1])
        se_std = float(f1["se"][f1["ns"] + 1])
        sd = scalers[cov]
        b_raw, eff = effect_string(b_std, meta, sd)
        z_w = b_std / se_std if (np.isfinite(se_std) and se_std > 0) else np.nan
        p_w = float(2 * stats.norm.sf(abs(z_w))) if np.isfinite(z_w) else np.nan
        if np.isfinite(se_std):
            lo_s, hi_s = b_std - 1.96 * se_std, b_std + 1.96 * se_std
            lo_r = lo_s / sd if meta["type"] == "cont" else lo_s
            hi_r = hi_s / sd if meta["type"] == "cont" else hi_s
            u = meta["unit"] if meta["type"] == "cont" else 1.0
            ci = (f"[{(np.exp(lo_r*u)-1)*100:+.2f}%, {(np.exp(hi_r*u)-1)*100:+.2f}%]")
        else:
            ci = "n/a"

        out.append({**base,
                    "Effect": eff, "CI95_effect": ci,
                    "coef_std": round(b_std, 6), "SE_std": round(se_std, 6),
                    "coef_per_unit": round(b_raw, 6) if np.isfinite(b_raw) else np.nan,
                    "Wald_z": round(z_w, 3) if np.isfinite(z_w) else np.nan,
                    "Wald_p": fmt_p(p_w),
                    "LR_chi2": round(LR, 3), "LR_p": fmt_p(p_lr), "_p_lr": p_lr,
                    "dAIC": round(dAIC, 2),
                    "improves": "Yes" if p_lr < ALPHA else "No",
                    "improves_strict": "Yes" if (p_lr < ALPHA and dAIC >= DAIC_MIN) else "No",
                    "converged": "Yes" if (f0["converged"] and f1["converged"]) else "check",
                    "warning": opt_warning})
    return out

rows = []
for pair in KEPT_PAIRS:
    g = dfk[dfk[STRATUM] == pair]
    rows += screen_pair(pair, g, pair_best[pair], COVARIATES)
screen = pd.DataFrame(rows)

# --- multiplicity: BH within each pair, and BH across the whole screen --------
screen["p_BH_within_pair"] = np.nan
for pair, idx in screen.groupby("Pair").groups.items():
    screen.loc[idx, "p_BH_within_pair"] = bh_adjust(screen.loc[idx, "_p_lr"].values)
screen["p_BH_overall"] = bh_adjust(screen["_p_lr"].values)
screen["survives_BH_within_pair"] = np.where(screen["p_BH_within_pair"] < ALPHA, "Yes",
                                    np.where(screen["p_BH_within_pair"].isna(), "n/a", "No"))
screen["survives_BH_overall"] = np.where(screen["p_BH_overall"] < ALPHA, "Yes",
                                np.where(screen["p_BH_overall"].isna(), "n/a", "No"))
screen["p_BH_within_pair"] = screen["p_BH_within_pair"].map(fmt_p)
screen["p_BH_overall"]     = screen["p_BH_overall"].map(fmt_p)

cols_order = ["Pair", "n_model", "Distribution", "Covariate", "Type", "Unit",
              "Effect", "CI95_effect", "coef_std", "SE_std", "coef_per_unit",
              "Wald_z", "Wald_p", "LR_chi2", "LR_p", "p_BH_within_pair", "p_BH_overall",
              "dAIC", "improves", "improves_strict", "survives_BH_within_pair",
              "survives_BH_overall", "converged", "n_dropped_missing", "warning"]
screen_out = screen.reindex(columns=cols_order)

dAIC_matrix = (screen.pivot(index="Pair", columns="Covariate", values="dAIC")
                     .reindex(index=KEPT_PAIRS, columns=COV_ORDER).astype(float).round(2))
eff_matrix  = (screen.pivot(index="Pair", columns="Covariate", values="Effect")
                     .reindex(index=KEPT_PAIRS, columns=COV_ORDER))

xl4 = os.path.join(TABLES, "04_covariate_screen.xlsx")
with pd.ExcelWriter(xl4, engine="openpyxl") as xl:
    screen_out.to_excel(xl,   sheet_name="Screening_long", index=False)
    dAIC_matrix.to_excel(xl,  sheet_name="dAIC_matrix")
    eff_matrix.to_excel(xl,   sheet_name="Effect_matrix")
print("Saved Excel 4:", xl4)

# --- heatmap -----------------------------------------------------------------
M = dAIC_matrix.values.astype(float)
fig, ax = plt.subplots(figsize=(1.05 * M.shape[1] + 3.4, 0.45 * M.shape[0] + 2.0))
vmax = np.nanpercentile(np.abs(M), 95) if np.isfinite(M).any() else 1
im = ax.imshow(M, cmap="RdYlGn", vmin=-vmax, vmax=vmax, aspect="auto")
ax.set_xticks(range(M.shape[1])); ax.set_xticklabels(dAIC_matrix.columns, rotation=40, ha="right")
ax.set_yticks(range(M.shape[0])); ax.set_yticklabels(dAIC_matrix.index)
for i in range(M.shape[0]):
    for j in range(M.shape[1]):
        if np.isfinite(M[i, j]):
            mark = "*" if str(screen_out.loc[
                (screen_out.Pair == dAIC_matrix.index[i]) &
                (screen_out.Covariate == dAIC_matrix.columns[j]), "improves_strict"].iloc[0]) == "Yes" else ""
            ax.text(j, i, f"{M[i, j]:.1f}{mark}", ha="center", va="center", fontsize=7)
ax.set_title(r"$\Delta$AIC (AIC$_{null}$ - AIC$_{cov}$);  * = p<0.05 and $\Delta$AIC$\geq$2")
ax.grid(False); fig.colorbar(im, ax=ax, shrink=0.8)
fig.savefig(os.path.join(GRAPHICS, "fig04_dAIC_heatmap.png")); plt.close(fig)

nbad = int((screen["warning"].fillna("") != "").sum())
if nbad:
    print(f"{nbad} model(s) hit an optimiser warning -- see the 'warning' column.")
dAIC_matrix

Saved Excel 4: D:\Headway\Tables\04_covariate_screen.xlsx


Covariate,Target_Speed_km/hr,Speed_Difference,Off_centeredness,Occupancy,Flow_pcu/hr/m,Site
Pair,,,,,,
BTW_following_4W,26.80,30.02,3.03,-1.14,-1.85,3.74
BTW_following_MT_3W,38.74,33.38,0.60,0.56,-1.24,12.74
BTW_following_NMT_3W,4.10,-1.57,-1.22,-1.60,-1.13,0.69
PR_following_MT_3W,16.08,1.13,-1.93,9.22,-1.95,0.86
BTW_following_MT_2W,2.31,4.99,-0.43,-1.99,-1.94,-0.51
PR_following_NMT_3W,-1.95,-0.16,-0.66,-1.45,0.47,-0.57
PR_following_4W,-1.99,-1.99,0.37,-1.79,-1.40,-1.84
BTW_following_NMT_2W,-0.43,7.69,-1.95,-2.00,-1.41,-1.73


In [11]:
# =============================================================================
# Cell 11 — STEP 6: Assemble the final covariate set per pair.
#           VIF is REPORTED alongside; it never removes a covariate.
#           -> Tables/05_final_covariate_sets.xlsx
# =============================================================================
# Selection rule for the compound model. Options:
#   "strict"  : LR p < ALPHA  AND  dAIC >= DAIC_MIN            (default)
#   "p"       : LR p < ALPHA
#   "bh"      : BH-adjusted p within pair < ALPHA
#   "all"     : every covariate enters (full model, no screening)
SELECTION_RULE = "strict"

_rule_col = {"strict": "improves_strict", "p": "improves",
             "bh": "survives_BH_within_pair", "all": None}[SELECTION_RULE]

def max_vif(frame, cols):
    if len(cols) < 2:
        return 1.0 if cols else np.nan
    v = compute_vif(frame, cols)["VIF"].values.astype(float)
    return round(float(np.nanmax(v)), 2) if np.isfinite(v).any() else np.nan

final, FINAL_SETS = [], {}
for pair in KEPT_PAIRS:
    g = dfk[dfk[STRATUM] == pair]
    sp = screen_out[screen_out["Pair"] == pair].copy()
    sp["_d"] = pd.to_numeric(sp["dAIC"], errors="coerce")
    sp = sp.sort_values("_d", ascending=False)

    if _rule_col is None:
        selected = [c for c in COV_ORDER if c in set(sp["Covariate"])]
    else:
        selected = sp.loc[sp[_rule_col] == "Yes", "Covariate"].tolist()

    estimable = sp.loc[sp["improves"] != "n/a", "Covariate"].tolist()
    not_estimable = [c for c in COV_ORDER if c not in estimable]
    FINAL_SETS[pair] = selected

    sel_cols = [COVARIATES[c]["col"] for c in selected]
    all_cols = [COVARIATES[c]["col"] for c in COV_ORDER]
    final.append(dict(
        Pair=pair, N=int((dfk[STRATUM] == pair).sum()), Distribution=pair_best[pair],
        ev_flag="Yes" if pair_best[pair] in FLAG_DISTS else "No",
        Screened=", ".join(COV_ORDER),
        Final_set=", ".join(selected) if selected else "none (intercept-only fit)",
        n_final=len(selected),
        Not_selected=", ".join([c for c in COV_ORDER if c not in selected]) or "-",
        Not_estimable=", ".join(not_estimable) or "-",
        max_VIF_final_set=max_vif(g, sel_cols),
        max_VIF_all_covariates=max_vif(g, all_cols),
        VIF_note="reported only - no covariate removed"))
final_sets = pd.DataFrame(final)

# per-pair VIF for exactly the covariates that entered each compound model
vif_final_long = []
for pair in KEPT_PAIRS:
    cols = [COVARIATES[c]["col"] for c in FINAL_SETS[pair]]
    if not cols:
        continue
    t = compute_vif(dfk[dfk[STRATUM] == pair], cols, FINAL_SETS[pair])
    t.insert(0, "Pair", pair)
    vif_final_long.append(t)
vif_final = pd.concat(vif_final_long, ignore_index=True) if vif_final_long else pd.DataFrame()

xl5 = os.path.join(TABLES, "05_final_covariate_sets.xlsx")
with pd.ExcelWriter(xl5, engine="openpyxl") as xl:
    final_sets.to_excel(xl, sheet_name="Final_model_spec", index=False)
    if len(vif_final):
        vif_final.to_excel(xl, sheet_name="VIF_of_final_sets", index=False)
    vif_pair_matrix.to_excel(xl, sheet_name="VIF_all_covs_by_pair")
print("Saved Excel 5:", xl5)
print(f"Selection rule = '{SELECTION_RULE}'. VIF is diagnostic only.")
hi = final_sets.loc[final_sets["max_VIF_final_set"] > VIF_REPORT_MO, "Pair"].tolist()
print("Pairs whose final set has max VIF > %g (retained, flagged in the paper): %s"
      % (VIF_REPORT_MO, hi if hi else "none"))
final_sets

Saved Excel 5: D:\Headway\Tables\05_final_covariate_sets.xlsx
Selection rule = 'strict'. VIF is diagnostic only.
Pairs whose final set has max VIF > 5 (retained, flagged in the paper): none


,Pair,N,Distribution,ev_flag,Screened,Final_set,n_final,Not_selected,Not_estimable,max_VIF_final_set,max_VIF_all_covariates,VIF_note
0,BTW_following_4W,250,gengamma,No,"Target_Speed_km/hr, Speed_Difference, Off_cent...","Speed_Difference, Target_Speed_km/hr, Site, Of...",4,"Occupancy, Flow_pcu/hr/m",-,2.11,2.16,reported only - no covariate removed
1,BTW_following_MT_3W,186,gamma,No,"Target_Speed_km/hr, Speed_Difference, Off_cent...","Target_Speed_km/hr, Speed_Difference, Site",3,"Off_centeredness, Occupancy, Flow_pcu/hr/m",-,2.25,2.33,reported only - no covariate removed
2,BTW_following_NMT_3W,119,gamma,No,"Target_Speed_km/hr, Speed_Difference, Off_cent...",Target_Speed_km/hr,1,"Speed_Difference, Off_centeredness, Occupancy,...",-,1.00,2.23,reported only - no covariate removed
3,PR_following_MT_3W,104,weibull_min,No,"Target_Speed_km/hr, Speed_Difference, Off_cent...","Target_Speed_km/hr, Occupancy",2,"Speed_Difference, Off_centeredness, Flow_pcu/h...",-,1.04,2.17,reported only - no covariate removed
4,BTW_following_MT_2W,73,invgauss,No,"Target_Speed_km/hr, Speed_Difference, Off_cent...","Speed_Difference, Target_Speed_km/hr",2,"Off_centeredness, Occupancy, Flow_pcu/hr/m, Site",-,1.00,3.26,reported only - no covariate removed
5,PR_following_NMT_3W,49,weibull_min,No,"Target_Speed_km/hr, Speed_Difference, Off_cent...",none (intercept-only fit),0,"Target_Speed_km/hr, Speed_Difference, Off_cent...",-,NaN,2.28,reported only - no covariate removed
6,PR_following_4W,43,weibull_min,No,"Target_Speed_km/hr, Speed_Difference, Off_cent...",none (intercept-only fit),0,"Target_Speed_km/hr, Speed_Difference, Off_cent...",-,NaN,3.14,reported only - no covariate removed
7,BTW_following_NMT_2W,41,invgauss,No,"Target_Speed_km/hr, Speed_Difference, Off_cent...",Speed_Difference,1,"Target_Speed_km/hr, Off_centeredness, Occupanc...",-,1.00,2.46,reported only - no covariate removed


---
### Step 7 — the compound model

v3's "final covariate set" was assembled from **univariate** screens and never
re-estimated. Everything below is a **joint** fit, so the coefficients are mutually
adjusted. `M2_full` (all covariates, no screening) is reported alongside so the paper
can show that the screened model is not an artefact of the selection rule.


In [12]:
# =============================================================================
# Cell 12 — STEP 7 (NEW): Multivariable "compound" AFT refit per pair
#   v3 stopped after univariate screening and never re-estimated the selected
#   covariates jointly, so no reported coefficient was adjusted for the others.
#   This cell fits, for every kept pair and under that pair's own distribution:
#       M0  intercept-only (null)
#       M1  the selected final set
#       M2  all covariates (full model, for comparison)
#   -> Tables/06_compound_models.xlsx  +  Graphics/fig06_forest.png
# =============================================================================
def fit_compound(pair, g, dist_name, cov_names, label):
    t, Z, scalers, used, ndrop = build_design(g, cov_names, COVARIATES)
    if len(t) < MIN_N_FIT:
        return None, []
    f0 = fit_aft(dist_name, t, np.empty((len(t), 0)))
    if f0 is None:
        return None, []
    fM = f0 if not used else fit_aft(dist_name, t, Z, warm=list(f0["x"]) + [0.0] * Z.shape[1])
    if fM is None:
        return None, []

    LR = max(2 * (fM["ll"] - f0["ll"]), 0.0)
    dfree = len(used)
    p_joint = float(stats.chi2.sf(LR, dfree)) if dfree > 0 else np.nan
    k = fM["nparam"]; n = len(t)
    summ = dict(Pair=pair, Model=label, Distribution=dist_name, n=n,
                Covariates=", ".join(used) if used else "none",
                n_params=k, LogLik=round(fM["ll"], 3),
                AIC=round(2 * k - 2 * fM["ll"], 2),
                BIC=round(k * np.log(n) - 2 * fM["ll"], 2),
                LR_vs_null=round(LR, 3), df=dfree, LR_p=fmt_p(p_joint),
                McFadden_R2=round(1 - fM["ll"] / f0["ll"], 4) if f0["ll"] != 0 else np.nan,
                converged="Yes" if fM["converged"] else "check")

    ns = fM["ns"]
    coefs = []
    for i, cov in enumerate(used):
        meta = COVARIATES[cov]; sd = scalers[cov]
        b_std = float(fM["x"][ns + 1 + i]); se = float(fM["se"][ns + 1 + i])
        b_raw, eff = effect_string(b_std, meta, sd)
        z_w = b_std / se if (np.isfinite(se) and se > 0) else np.nan
        p_w = float(2 * stats.norm.sf(abs(z_w))) if np.isfinite(z_w) else np.nan
        if np.isfinite(se):
            u = meta["unit"] if meta["type"] == "cont" else 1.0
            s = sd if meta["type"] == "cont" else 1.0
            lo = (np.exp((b_std - 1.96 * se) / s * u) - 1) * 100
            hi = (np.exp((b_std + 1.96 * se) / s * u) - 1) * 100
        else:
            lo = hi = np.nan
        coefs.append(dict(Pair=pair, Model=label, Distribution=dist_name, n=n,
                          Covariate=cov, Type=meta["type"], Unit=meta["label"],
                          coef_std=round(b_std, 6), SE_std=round(se, 6),
                          coef_per_unit=round(b_raw, 6) if np.isfinite(b_raw) else np.nan,
                          Effect=eff,
                          CI_low_pct=round(lo, 2), CI_high_pct=round(hi, 2),
                          Wald_z=round(z_w, 3) if np.isfinite(z_w) else np.nan,
                          Wald_p=fmt_p(p_w), sig=stars(p_w), _p=p_w))
    return summ, coefs

model_summaries, coef_rows = [], []
for pair in KEPT_PAIRS:
    g = dfk[dfk[STRATUM] == pair]; dn = pair_best[pair]
    s0, _ = fit_compound(pair, g, dn, [], "M0_null")
    if s0: model_summaries.append(s0)
    s1, c1 = fit_compound(pair, g, dn, FINAL_SETS[pair], "M1_selected")
    if s1: model_summaries.append(s1); coef_rows += c1
    s2, c2 = fit_compound(pair, g, dn, COV_ORDER, "M2_full")
    if s2: model_summaries.append(s2); coef_rows += c2

compound_summary = pd.DataFrame(model_summaries)
compound_coefs   = pd.DataFrame(coef_rows)

if len(compound_coefs):
    for (pair, mod), idx in compound_coefs.groupby(["Pair", "Model"]).groups.items():
        compound_coefs.loc[idx, "p_BH_within_model"] = bh_adjust(compound_coefs.loc[idx, "_p"].values)
    compound_coefs["p_BH_within_model"] = compound_coefs["p_BH_within_model"].map(fmt_p)
    compound_coefs = compound_coefs.drop(columns=["_p"])

# model comparison: does the selected model beat the full model on AIC?
cmp_tbl = (compound_summary.pivot(index="Pair", columns="Model", values="AIC")
                           .reindex(index=KEPT_PAIRS).round(2))
cmp_tbl["best_by_AIC"] = cmp_tbl.idxmin(axis=1)

# publication-ready coefficient table for the selected models
pub = compound_coefs[compound_coefs["Model"] == "M1_selected"].copy() if len(compound_coefs) else pd.DataFrame()
if len(pub):
    pub["Estimate (95% CI)"] = pub.apply(
        lambda r: f"{r['Effect'].split('  ')[0]} [{r['CI_low_pct']:+.1f}, {r['CI_high_pct']:+.1f}]"
        if pd.notna(r["CI_low_pct"]) else r["Effect"], axis=1)
    pub = pub[["Pair", "Distribution", "n", "Covariate", "Unit",
               "Estimate (95% CI)", "Wald_p", "p_BH_within_model", "sig"]]

xl6 = os.path.join(TABLES, "06_compound_models.xlsx")
with pd.ExcelWriter(xl6, engine="openpyxl") as xl:
    compound_summary.to_excel(xl, sheet_name="Model_summary", index=False)
    if len(pub):
        pub.to_excel(xl,          sheet_name="Publication_table", index=False)
    if len(compound_coefs):
        compound_coefs.to_excel(xl, sheet_name="Coefficients_all", index=False)
    cmp_tbl.to_excel(xl,          sheet_name="AIC_comparison")
print("Saved Excel 6:", xl6)

# --- forest plot of the selected-model coefficients ---------------------------
fp = compound_coefs[compound_coefs["Model"] == "M1_selected"].dropna(subset=["CI_low_pct"]) \
     if len(compound_coefs) else pd.DataFrame()
if len(fp):
    fp = fp.sort_values(["Pair", "Covariate"]).reset_index(drop=True)
    lab = [f"{r.Pair} | {r.Covariate}" for r in fp.itertuples()]
    ctr = [(np.exp(r.coef_per_unit * (COVARIATES[r.Covariate]['unit']
            if COVARIATES[r.Covariate]['type'] == 'cont' else 1.0)) - 1) * 100
           for r in fp.itertuples()]
    y = np.arange(len(fp))
    fig, ax = plt.subplots(figsize=(7.2, 0.32 * len(fp) + 2.0))
    ax.hlines(y, fp["CI_low_pct"], fp["CI_high_pct"], color="#4C72B0", lw=2)
    ax.plot(ctr, y, "o", color="#C44E52", ms=4)
    ax.axvline(0, color="k", lw=1)
    ax.set_yticks(y); ax.set_yticklabels(lab, fontsize=7); ax.invert_yaxis()
    ax.set_xlabel("% change in the scale parameter (\u2248 % change in median headway)")
    ax.set_title("Compound AFT models: adjusted covariate effects with 95% CI")
    fig.savefig(os.path.join(GRAPHICS, "fig06_forest.png")); plt.close(fig)
    print("Saved:", os.path.join(GRAPHICS, "fig06_forest.png"))

compound_summary

Saved Excel 6: D:\Headway\Tables\06_compound_models.xlsx
Saved: D:\Headway\Graphics\fig06_forest.png


,Pair,Model,Distribution,n,Covariates,n_params,LogLik,AIC,BIC,LR_vs_null,df,LR_p,McFadden_R2,converged
0,BTW_following_4W,M0_null,gengamma,250,none,3,-308.811,623.62,634.19,0.000,0,n/a,0.0000,Yes
1,BTW_following_4W,M1_selected,gengamma,250,"Speed_Difference, Target_Speed_km/hr, Site, Of...",7,-275.956,565.91,590.56,65.710,4,<0.001,0.1064,Yes
2,BTW_following_4W,M2_full,gengamma,250,"Target_Speed_km/hr, Speed_Difference, Off_cent...",9,-275.424,568.85,600.54,66.775,6,<0.001,0.1081,Yes
3,BTW_following_MT_3W,M0_null,gamma,186,none,2,-219.243,442.49,448.94,0.000,0,n/a,0.0000,Yes
4,BTW_following_MT_3W,M1_selected,gamma,186,"Target_Speed_km/hr, Speed_Difference, Site",5,-189.019,388.04,404.17,60.447,3,<0.001,0.1379,Yes
5,BTW_following_MT_3W,M2_full,gamma,186,"Target_Speed_km/hr, Speed_Difference, Off_cent...",8,-188.241,392.48,418.29,62.003,6,<0.001,0.1414,Yes
6,BTW_following_NMT_3W,M0_null,gamma,119,none,2,-147.005,298.01,303.57,0.000,0,n/a,0.0000,Yes
7,BTW_following_NMT_3W,M1_selected,gamma,119,Target_Speed_km/hr,3,-143.956,293.91,302.25,6.098,1,0.0135,0.0207,Yes
8,BTW_following_NMT_3W,M2_full,gamma,119,"Target_Speed_km/hr, Speed_Difference, Off_cent...",8,-141.401,298.80,321.04,11.207,6,0.0822,0.0381,Yes
9,PR_following_MT_3W,M0_null,weibull_min,104,none,2,-137.323,278.65,283.93,0.000,0,n/a,0.0000,Yes


In [13]:
# =============================================================================
# Cell 13 — Distribution-fit graphics (v3 produced no figures at all)
#   fig03a  histogram + fitted density, one panel per pair
#   fig03b  empirical vs fitted CDF
#   fig03c  Q-Q plot of the selected distribution
# =============================================================================
def _grid(n, ncol=3):
    nrow = int(np.ceil(n / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(3.6 * ncol, 2.7 * nrow), squeeze=False)
    return fig, axes.ravel(), nrow * ncol

pairs_plot = KEPT_PAIRS
fig, axes, slots = _grid(len(pairs_plot))
for ax, pair in zip(axes, pairs_plot):
    t = dfk.loc[dfk[STRATUM] == pair, OUTCOME].values.astype(float)
    dn = pair_best[pair]; dist = getattr(stats, dn); prm = pair_best_params[pair]
    ax.hist(t, bins=min(25, max(8, len(t) // 12)), density=True,
            color="#B0C4DE", edgecolor="white", lw=.4)
    xs = np.linspace(max(t.min() * 0.6, 1e-3), t.max() * 1.05, 400)
    ax.plot(xs, dist.pdf(xs, *prm), color="#C44E52", lw=1.6)
    ax.set_title(f"{pair}\n{dn} (n={len(t)})", fontsize=8)
    ax.set_xlabel("time headway (s)"); ax.set_ylabel("density")
for ax in axes[len(pairs_plot):]:
    ax.axis("off")
fig.tight_layout(); fig.savefig(os.path.join(GRAPHICS, "fig03a_pdf_overlay.png")); plt.close(fig)

fig, axes, slots = _grid(len(pairs_plot))
for ax, pair in zip(axes, pairs_plot):
    t = np.sort(dfk.loc[dfk[STRATUM] == pair, OUTCOME].values.astype(float))
    dn = pair_best[pair]; dist = getattr(stats, dn); prm = pair_best_params[pair]
    ecdf = np.arange(1, len(t) + 1) / len(t)
    ax.step(t, ecdf, where="post", color="#4C72B0", lw=1.2, label="empirical")
    ax.plot(t, dist.cdf(t, *prm), color="#C44E52", lw=1.4, label="fitted")
    ax.set_title(f"{pair} | {dn}", fontsize=8)
    ax.set_xlabel("time headway (s)"); ax.set_ylabel("F(t)")
    ax.legend(fontsize=6, frameon=False)
for ax in axes[len(pairs_plot):]:
    ax.axis("off")
fig.tight_layout(); fig.savefig(os.path.join(GRAPHICS, "fig03b_cdf_overlay.png")); plt.close(fig)

fig, axes, slots = _grid(len(pairs_plot))
for ax, pair in zip(axes, pairs_plot):
    t = np.sort(dfk.loc[dfk[STRATUM] == pair, OUTCOME].values.astype(float))
    dn = pair_best[pair]; dist = getattr(stats, dn); prm = pair_best_params[pair]
    q = (np.arange(1, len(t) + 1) - 0.5) / len(t)
    theo = dist.ppf(q, *prm)
    ok = np.isfinite(theo)
    ax.plot(theo[ok], t[ok], "o", ms=2.4, color="#4C72B0", alpha=.7)
    lim = [min(theo[ok].min(), t.min()), max(theo[ok].max(), t.max())]
    ax.plot(lim, lim, "--", color="#C44E52", lw=1)
    ax.set_title(f"{pair} | {dn}", fontsize=8)
    ax.set_xlabel("theoretical quantile"); ax.set_ylabel("empirical quantile")
for ax in axes[len(pairs_plot):]:
    ax.axis("off")
fig.tight_layout(); fig.savefig(os.path.join(GRAPHICS, "fig03c_qq.png")); plt.close(fig)

for f in ["fig03a_pdf_overlay.png", "fig03b_cdf_overlay.png", "fig03c_qq.png"]:
    print("Saved:", os.path.join(GRAPHICS, f))

Saved: D:\Headway\Graphics\fig03a_pdf_overlay.png
Saved: D:\Headway\Graphics\fig03b_cdf_overlay.png
Saved: D:\Headway\Graphics\fig03c_qq.png


In [14]:
# =============================================================================
# Cell 14 — Reproducibility stamp + output manifest (paste into the paper's
#           "Data and Methods" / supplementary material)
# =============================================================================
import scipy
provenance = pd.DataFrame([
    dict(Item="run timestamp",        Value=datetime.now().strftime("%Y-%m-%d %H:%M:%S")),
    dict(Item="python",               Value=platform.python_version()),
    dict(Item="numpy / scipy / pandas", Value=f"{np.__version__} / {scipy.__version__} / {pd.__version__}"),
    dict(Item="RNG seed",             Value=RNG_SEED),
    dict(Item="data file",            Value=DATA_PATH),
    dict(Item="observations (raw / analysed)", Value=f"{n0} / {len(dfk)}"),
    dict(Item="outcome",              Value=OUTCOME),
    dict(Item="stratum",              Value=STRATUM),
    dict(Item="pairs kept / dropped", Value=f"{len(KEPT_PAIRS)} / {len(DROP_PAIRS)}"),
    dict(Item="covariates",           Value=", ".join(COV_ORDER)),
    dict(Item="excluded by design",   Value=", ".join([c for c in EXCLUDE_FROM_COVARIATES if c in df.columns])),
    dict(Item="candidate distributions", Value=", ".join(CANDIDATE_DISTS)),
    dict(Item="loc fixed at 0",       Value=str(FIT_FLOC0)),
    dict(Item="truncation",           Value=str(TRUNC)),
    dict(Item="bootstrap KS reps",    Value=N_BOOT_KS),
    dict(Item="selection rule",       Value=SELECTION_RULE),
    dict(Item="alpha / dAIC_min",     Value=f"{ALPHA} / {DAIC_MIN}"),
    dict(Item="VIF policy",           Value="reported only; no covariate removed"),
])

xl0 = os.path.join(TABLES, "00_provenance.xlsx")
with pd.ExcelWriter(xl0, engine="openpyxl") as xl:
    provenance.to_excel(xl, sheet_name="Provenance", index=False)
    retention.to_excel(xl,  sheet_name="Pair_retention", index=False)
print("Saved:", xl0)

print("\nTables written to", TABLES)
for f in sorted(os.listdir(TABLES)):
    if f.endswith(".xlsx"):
        print("   ", f)
print("\nFigures written to", GRAPHICS)
for f in sorted(os.listdir(GRAPHICS)):
    if f.endswith(".png"):
        print("   ", f)
provenance

Saved: D:\Headway\Tables\00_provenance.xlsx

Tables written to D:\Headway\Tables
    00_provenance.xlsx
    01_inferential_stats.xlsx
    02_vif_multicollinearity.xlsx
    03_distribution_fits.xlsx
    04_covariate_screen.xlsx
    04_covariate_screen_ownbest.xlsx
    05_final_covariate_sets.xlsx
    06_compound_models.xlsx
    06_model_fit_verification.xlsx
    07_pairwise_gof_distributions.xlsx
    12. prep_leading_grouping.xlsx
    Table_Sample_Description.xlsx

Figures written to D:\Headway\Graphics
    Fig_TH_Speed_KDE_ECDF_collage.png
    fig02a_vif_global.png
    fig02b_corr_covariates.png
    fig03a_pdf_overlay.png
    fig03b_cdf_overlay.png
    fig03c_qq.png
    fig04_dAIC_heatmap.png
    fig06_forest.png
    gof_BTW_following_4W.png
    gof_BTW_following_MT_2W.png
    gof_BTW_following_MT_3W.png
    gof_BTW_following_NMT_2W.png
    gof_BTW_following_NMT_3W.png
    gof_PR_following_4W.png
    gof_PR_following_MT_3W.png
    gof_PR_following_NMT_3W.png
    gof_all_pairs_grid.png


,Item,Value
0,run timestamp,2026-07-24 11:07:09
1,python,3.14.6
2,numpy / scipy / pandas,2.4.6 / 1.18.0 / 2.3.3
3,RNG seed,20250101
4,data file,D:\Headway\data3.xlsx
5,observations (raw / analysed),898 / 865
6,outcome,Time_Headway
7,stratum,Pair
8,pairs kept / dropped,8 / 2
9,covariates,"Target_Speed_km/hr, Speed_Difference, Off_cent..."
